# ***Notebook for data preparation for YOLO model training***
- Bounding Box for box detection (object localization)
- Oriented Bounding Box for oriented object detection and loccalization
- Instance Masks for instance segmentation

Note: The dataset is Very High Resolution drone imagery obtained from [Open Aerial Maps](https://map.openaerialmap.org/#/33.47534179687499,0.26367094433665017,6?_k=xixa3o)

In [ ]:
!uv pip install -U imagecodecs
!uv pip install leafmap
!uv pip install localtileserver
!uv pip install kagglehub
!pip install --upgrade numpy
!pip install --upgrade scikit-image

In [1]:
import kagglehub
import os
from google.colab import drive
import sys

# Connect Google drive for saving and accessing files

In [2]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## **Connect the local scrit folder to use cutom functions**

In [3]:
import sys
sys.path.append("/content/drive/MyDrive/Deep-Learning-Building-Extraction-master/scripts")

### **Load custom preprocessing scripts**
The folowing cell imports scripts from the local folder. The scripts are usefull for tiling the input image

In [4]:
import prepare_yolo_mask as masker
from train_plotter import plot_yolo_bbox_dataset, plot_yolo_segmentation_dataset, plot_yolo_obb_dataset
from tiling_and_visualization import tile_raster, plot_binary_mask, tile_visualize

## **Access data from remote server**

We will access the data stored in Kaggle using its own access API as follows

In [ ]:
data_path = kagglehub.dataset_download("getachewworkineh/kakuma-ceos-training")
print("Path to dataset files:", data_path)
print("Datasets in the folder: ", os.listdir(data_path))

### ***Proper tile selection through visualization***

In [ ]:
train_file = 'train.tif'
raster_file = os.path.join(data_path, train_file)
tile_height = 640
tile_width = 640
stride_y = 630
stride_x = 630
tile_visualize(raster_file=raster_file,
                   tile_height=tile_height,
                   tile_width=tile_width,
                   stride_y=stride_y,
                   stride_x=stride_x,
                   with_tiles=True)

### **Specify the input shapefile or mask and raster image and perform image tiling or create sample chips**
 - tile training data

In [ ]:
input_raster = os.path.join(data_path, 'train.tif')
input_mask_file = os.path.join(data_path, 'training.geojson')
output_train_directory = "/content/drive/MyDrive/raw_dataset/train"


tile_size = 640
stride = 640
background_value = 0

tile_raster(
        input_raster=input_raster,
        output_dir=output_train_directory,
        input_mask_file=input_mask_file,
        tile_size=tile_size,
        stride=stride,
        background_value=background_value)

## Visualize tiled training images and corresponding masks

In [ ]:
plot_binary_mask(file_path=output_train_directory, n_samples=12)

  - tile validation data

In [ ]:
input_raster = os.path.join(data_path, 'valid.tif')
input_mask_file = os.path.join(data_path, 'validation.geojson')
output_valid_directory = "/content/drive/MyDrive/raw_dataset/valid"


tile_size = 640
stride = 640
background_value = 0

tile_raster(
        input_raster=input_raster,
        output_dir=output_valid_directory,
        input_mask_file=input_mask_file,
        tile_size=tile_size,
        stride=stride,
        background_value=background_value)

## Visualize tiled validation images and corresponding masks

In [ ]:
plot_binary_mask(file_path=output_valid_directory, n_samples=12)

### **Specify the path that contains the tiles path and convert classified tiles into YOLO sample format**

- Specify the input directory where the tiled chips are saved
Specify the output directory where samples prepared as YOLO format will be saved

In [13]:
input_path_train = output_train_directory # "/content/drive/MyDrive/dataset/aoi_112_with_yolo_mask"
input_path_valid = output_valid_directory #  "/content/drive/MyDrive/dataset/aoi_112_with_yolo_mask"


output_path_bbox = "/content/drive/MyDrive/yolo_dataset/bbox"        # for bounding box detection
output_path_obbox = "/content/drive/MyDrive/yolo_dataset/obbox"      # for oriented bounding box detection
output_path_segment = "/content/drive/MyDrive/yolo_dataset/segment"  # for instance segmentation

### ***Bounding Box Format***

In [ ]:
# train
masker.prepare_yolo_data(data_dir=input_path_train,
                          out_dir=output_path_bbox,
                          ext='tif',
                          part="train",
                          output="bbox", # annotation format
                          upsample=False,
                          tobits=False,
                         write_config=True)
# Valid
masker.prepare_yolo_data(data_dir=input_path_valid,
                          out_dir=output_path_bbox,
                          ext='tif',
                          part="valid",
                          output="bbox",
                          upsample=False,
                          tobits=False,
                         write_config=False)

### ***Oriented bounding box***

In [ ]:
# train
masker.prepare_yolo_data(data_dir=input_path_train,
                          out_dir=output_path_obbox,
                          ext='tif',
                          part="train",
                          output="obbox",
                          upsample=False,
                          tobits=False,
                         write_config=True)
# valid
masker.prepare_yolo_data(data_dir=input_path_valid,
                          out_dir=output_path_obbox,
                          ext='tif',
                          part="valid",
                          output="obbox",
                          upsample=False,
                          tobits=False,
                         write_config=False)

### ***Instance segmentation masks***

In [ ]:
# Train
masker.prepare_yolo_data(data_dir=input_path_train,
                          out_dir=output_path_segment,
                          ext='tif',
                          part="train",
                          output="segment",
                          upsample=False,
                          tobits=False,
                         write_config=True)
# Valid

masker.prepare_yolo_data(data_dir=input_path_valid,
                          out_dir=output_path_segment,
                          ext='tif',
                          part="valid",
                          output="segment",
                          upsample=False,
                          tobits=False,
                         write_config=False)

### ***Visualize samples converted into YOLO Bounding Box format***

In [ ]:
class_names = ['building']
dataset_path = output_path_bbox + "/dataset/train"
plot_yolo_bbox_dataset(
    dataset_path=dataset_path,
    num_samples=9,
    class_names=class_names,
    ext="*.tif"
)

### ***Visualize samples converted into YOLO Oriented Bounding Box format***

In [ ]:
dataset_path = output_path_obbox + "/dataset/train"
class_nmes = ["building"]

plot_yolo_obb_dataset(
        dataset_path=dataset_path,
        class_names=class_nmes,
        num_samples=9,
        ext="*.tif"
    )

### ***Visualize samples converted into YOLO instance segmentation format***

In [ ]:
dataset_path = output_path_segment + "/dataset/valid"
class_names = ["building"]
print("Plotting dataset samples...")
plot_yolo_segmentation_dataset(dataset_path=dataset_path,
                               num_samples=9,
                               class_names=class_names,
                               ext="*.tif")